# Does RL teach a small LM new math, or just sharpen what it already knows?

**GRPO with verifiable rewards on Qwen2.5-0.5B-Instruct, built to run on a free Colab T4.**

Research questions:
1. **Does it improve?** Greedy accuracy on held-out problems from the training distribution (`test_id`).
2. **Does it generalize?** Accuracy on longer expressions (`test_ood_length`) and bigger numbers (`test_ood_digits`) the model never trained on.
3. **Teach or elicit?** If RL only *sharpens* the base model's distribution, pass@1 rises but pass@k at large k stays flat or drops (Yue et al., 2025, *Does RL Really Incentivize Reasoning Capacity Beyond the Base Model?*). We measure pass@1..8 before and after training.

Runtime: a T4 GPU (Colab or Kaggle). On Kaggle, use **Save & Run All** so long runs finish without the tab open. The baseline takes about 45 min; each experiment trains for several hours, then evaluates in about 20 min.

## 1. Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU; Trainer's auto multi-GPU breaks GRPO generation (Kaggle T4 x2)
%pip install -q "trl>=0.19" peft accelerate datasets matplotlib
%pip uninstall -y -q torchao               # Colab/Kaggle preinstall an old torchao that recent peft rejects

In [ ]:
# Save results and checkpoints to Google Drive so a Colab disconnect doesn't lose them.
# Kaggle ships a google.colab stub whose mount() raises, so catch everything.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = "/content/drive/MyDrive"
except Exception:
    RUN_ROOT = "runs"          # Kaggle / local Jupyter
print("Saving runs under:", RUN_ROOT)

In [ ]:
%%writefile rlmath.py
"""rlmath: GRPO with verifiable rewards on synthetic arithmetic.

Data generation with in-distribution / out-of-distribution splits,
reward functions, and pass@k evaluation. Shared by the Colab notebook.
"""
from __future__ import annotations

import json
import math
import random
import re
from dataclasses import dataclass, asdict

import numpy as np

# ---------------------------------------------------------------- data

SYSTEM_PROMPT = (
    "You are a careful math assistant. Respond in English only. "
    "Reason step by step inside <think> </think> tags, then give only the final "
    "integer inside <answer> </answer> tags."
)
# Every response is started with this text, so the model continues from inside the
# think block. Prompting alone was unreliable: Qwen switched to Chinese tags (Run 0)
# or dropped tags entirely when shown an example (Run 2). Same trick as TinyZero.
PREFILL = "<think>\n"


@dataclass(frozen=True)
class Level:
    name: str
    n_operands: tuple[int, int]   # inclusive range
    digits: tuple[int, int]       # inclusive range of digits per operand
    ops: str = "+-*"
    max_mults: int = 1            # cap multiplications so answers stay sane


# Train on easy-to-medium; hold out longer expressions and bigger numbers.
LEVELS = {
    "train":      Level("train",      (2, 3), (1, 2)),
    "ood_length": Level("ood_length", (4, 5), (1, 2), max_mults=2),
    "ood_digits": Level("ood_digits", (2, 3), (3, 3)),
    # Harder training distribution: 3-4 operands. Overlaps test_ood_length at 4 operands,
    # so 5-operand problems are the clean OOD slice for this experiment.
    "hard": Level("hard", (3, 4), (1, 2), max_mults=2),
}


def _rand_operand(rng: random.Random, digits: tuple[int, int]) -> int:
    d = rng.randint(*digits)
    return rng.randint(10 ** (d - 1) if d > 1 else 1, 10 ** d - 1)


def make_expression(rng: random.Random, level: Level) -> tuple[str, int]:
    n = rng.randint(*level.n_operands)
    nums = [_rand_operand(rng, level.digits) for _ in range(n)]
    ops, mults = [], 0
    for _ in range(n - 1):
        choices = [o for o in level.ops if not (o == "*" and mults >= level.max_mults)]
        o = rng.choice(choices)
        mults += o == "*"
        ops.append(o)
    expr = str(nums[0])
    for o, x in zip(ops, nums[1:]):
        expr += f" {o} {x}"
    # Only digits, spaces and + - * reach eval.
    assert re.fullmatch(r"[\d +\-*]+", expr)
    return expr, int(eval(expr))


def make_split(level: Level, n: int, seed: int, exclude: set[str] | None = None) -> list[dict]:
    rng = random.Random(seed)
    exclude = exclude or set()
    seen, rows = set(), []
    tries = 0
    while len(rows) < n:
        tries += 1
        if tries > n * 50:
            raise RuntimeError(f"could not draw {n} unique problems for {level.name}")
        expr, ans = make_expression(rng, level)
        if expr in seen or expr in exclude:
            continue
        seen.add(expr)
        rows.append({"expression": expr, "answer": ans, "level": level.name,
                     "n_operands": expr.count(" ") // 2 + 1})
    return rows


def to_prompt(row: dict) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Compute: {row['expression']}"},
    ]


def render_prompt(tok, row: dict) -> str:
    """Chat-formatted prompt string ending with PREFILL. Used for both training and eval."""
    return tok.apply_chat_template(row["prompt"], tokenize=False,
                                   add_generation_prompt=True) + PREFILL


def build_datasets(n_train=4000, n_eval=300, seed=0) -> dict[str, list[dict]]:
    train = make_split(LEVELS["train"], n_train, seed)
    train_exprs = {r["expression"] for r in train}
    splits = {
        "train": train,
        "test_id": make_split(LEVELS["train"], n_eval, seed + 1, exclude=train_exprs),
        "test_ood_length": make_split(LEVELS["ood_length"], n_eval, seed + 2),
        "test_ood_digits": make_split(LEVELS["ood_digits"], n_eval, seed + 3),
    }
    # Built after the test sets (which stay identical to earlier runs) and excluding
    # every test expression, so no test problem appears in the hard training set.
    test_exprs = {r["expression"] for k, v in splits.items() if k.startswith("test") for r in v}
    splits["train_hard"] = make_split(LEVELS["hard"], n_train, seed + 10, exclude=test_exprs)
    for rows in splits.values():
        for r in rows:
            r["prompt"] = to_prompt(r)
    return splits


# ------------------------------------------------------------- parsing

# Qwen sometimes switches tags to Chinese (<思考>/<答案>) and may omit the closing tag.
# The parser accepts both, so a correct answer is never scored wrong because of tag style
# (that bug made Run 0 punish reasoning and collapse to bare answers).
_ANSWER_RE = re.compile(r"<(?:answer|答案)>\s*(.*?)\s*(?:</(?:answer|答案)>|$)", re.S)
# The format reward stays strict: English think block followed by answer block.
_FORMAT_RE = re.compile(r"<think>.*?</think>\s*<answer>.*?</answer>", re.S)


def completion_text(c) -> str:
    """Full response text: PREFILL plus what the model generated.

    TRL passes completions as strings for string prompts (what we use) and as
    [{'role':..,'content':..}] for chat prompts; both are handled.
    """
    if isinstance(c, list):
        c = c[0]["content"] if c else ""
    return c if c.startswith(PREFILL) else PREFILL + c


def extract_answer(text: str) -> int | None:
    m = _ANSWER_RE.findall(text)
    if not m:
        return None
    nums = re.findall(r"-?\d[\d,]*", m[-1])
    if not nums:
        return None
    try:
        return int(nums[-1].replace(",", ""))
    except ValueError:
        return None


def extract_answer_lenient(text: str) -> int | None:
    """Tagged answer if present, else the last integer (used by correctness_reward).

    Lets us separate "got the math wrong" from "got the format wrong".
    """
    a = extract_answer(text)
    if a is not None:
        return a
    nums = re.findall(r"-?\d[\d,]*", text)
    try:
        return int(nums[-1].replace(",", "")) if nums else None
    except ValueError:
        return None


def output_style(text: str) -> str:
    """Coarse label of how the model formatted its response."""
    if _FORMAT_RE.search(text):
        return "format_ok"
    if _ANSWER_RE.search(text):
        return "answer_tag_only"
    return "no_answer_tag"


# ------------------------------------------------------------- rewards
# Signature required by TRL: (prompts, completions, **dataset_columns) -> list[float]

def correctness_reward(prompts, completions, answer, **kw) -> list[float]:
    """Lenient: tagged answer if present, else the last integer in the response.

    The strict tag-only version scored correct math as wrong whenever the model closed
    its reasoning with </thought> instead of </think><answer>, which happened on 99% of
    base-model outputs (Run 3: strict 0.003 vs lenient 0.86). Format is taught separately
    by format_reward, so correctness measures math only.
    """
    return [1.0 if extract_answer_lenient(completion_text(c)) == int(a) else 0.0
            for c, a in zip(completions, answer)]


def correctness_reward_strict(prompts, completions, answer, **kw) -> list[float]:
    """Tag-only version, kept for the ablation."""
    return [1.0 if extract_answer(completion_text(c)) == int(a) else 0.0
            for c, a in zip(completions, answer)]


def format_reward(prompts, completions, **kw) -> list[float]:
    return [0.1 if _FORMAT_RE.search(completion_text(c)) else 0.0 for c in completions]


# ---------------------------------------------------------- evaluation

def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased estimator from Chen et al. (2021), Codex paper."""
    if n - c < k:
        return 1.0
    return 1.0 - float(np.prod(1.0 - k / np.arange(n - c + 1, n + 1)))


def generate(model, tok, rows, n_samples=1, temperature=0.0, max_new_tokens=256,
             batch_size=32) -> list[list[str]]:
    """Return n_samples completions per row (HF generate, left-padded batches)."""
    import torch
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    texts = [render_prompt(tok, r) for r in rows]
    out: list[list[str]] = []
    model.eval()
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], return_tensors="pt", padding=True).to(model.device)
        gen_kw = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id,
                      num_return_sequences=n_samples)
        if temperature > 0:
            gen_kw.update(do_sample=True, temperature=temperature, top_p=1.0)
        else:
            gen_kw.update(do_sample=False)
        with torch.no_grad():
            g = model.generate(**enc, **gen_kw)
        dec = [PREFILL + d for d in
               tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
        for j in range(0, len(dec), n_samples):
            out.append(dec[j:j + n_samples])
    return out


def evaluate(model, tok, rows, n_samples=8, ks=(1, 2, 4, 8), temperature=0.8,
             max_new_tokens=256, batch_size=32, greedy=True) -> dict:
    """Greedy accuracy + sampled pass@k, plus per-row records for error analysis."""
    res: dict = {"n": len(rows)}
    records = [dict(expression=r["expression"], answer=r["answer"],
                    n_operands=r["n_operands"]) for r in rows]
    if greedy:
        g = generate(model, tok, rows, 1, 0.0, max_new_tokens, batch_size)
        for rec, (txt,) in zip(records, g):
            pred = extract_answer(txt)
            rec.update(greedy_text=txt, greedy_pred=pred, greedy_correct=pred == rec["answer"],
                       greedy_correct_lenient=extract_answer_lenient(txt) == rec["answer"],
                       greedy_format=bool(_FORMAT_RE.search(txt)), greedy_len=len(txt),
                       style=output_style(txt))
        res["greedy_acc"] = float(np.mean([r["greedy_correct"] for r in records]))
        res["lenient_acc"] = float(np.mean([r["greedy_correct_lenient"] for r in records]))
        res["format_rate"] = float(np.mean([r["greedy_format"] for r in records]))
        res["answer_tag_rate"] = float(np.mean([r["style"] != "no_answer_tag" for r in records]))
        res["mean_chars"] = float(np.mean([r["greedy_len"] for r in records]))
    if n_samples:
        s = generate(model, tok, rows, n_samples, temperature, max_new_tokens,
                     max(1, batch_size // n_samples))
        for rec, samples in zip(records, s):
            rec["n_correct"] = sum(extract_answer(t) == rec["answer"] for t in samples)
            rec["n_correct_lenient"] = sum(extract_answer_lenient(t) == rec["answer"] for t in samples)
        for k in ks:
            if k <= n_samples:
                res[f"pass@{k}"] = float(np.mean(
                    [pass_at_k(n_samples, r["n_correct"], k) for r in records]))
        # Format-independent capability: the fairer measure for "teach vs elicit".
        for k in ks:
            if k <= n_samples:
                res[f"lenient_pass@{k}"] = float(np.mean(
                    [pass_at_k(n_samples, r["n_correct_lenient"], k) for r in records]))
    res["records"] = records
    return res


def summarize(results: dict[str, dict]) -> str:
    """results: {split_name: evaluate(...) output} -> markdown table."""
    keys = [k for k in next(iter(results.values())) if k not in ("records", "n")]
    lines = ["| split | " + " | ".join(keys) + " |",
             "|---" * (len(keys) + 1) + "|"]
    for split, r in results.items():
        lines.append(f"| {split} | " + " | ".join(f"{r[k]:.3f}" for k in keys) + " |")
    return "\n".join(lines)


def save_results(path: str, results: dict) -> None:
    with open(path, "w") as f:
        json.dump(results, f, indent=1, default=str)

In [ ]:
import os, json, time, torch, numpy as np, matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
import rlmath

SMOKE = os.environ.get("SMOKE") == "1"          # tiny CPU run used only to test the pipeline
MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
OUT_DIR = os.environ.get("OUT_DIR", f"{RUN_ROOT}/grpo_arith_experiments")
SEED = 0

# Experiment knobs (the ablations live here)
CFG = dict(
    n_train=4000, n_eval=300,
    max_steps=250, lr=2e-5, beta=0.0,            # beta>0 adds a KL penalty to the base model
    num_generations=8, per_device_bs=8, grad_accum=2,
    max_completion_length=256, temperature=1.0,
    lora_r=16, lora_alpha=32,
    use_format_reward=True,
    eval_samples=8,                               # samples per problem for pass@k
)
if SMOKE:
    CFG.update(n_train=64, n_eval=8, max_steps=2, num_generations=4, per_device_bs=4,
               grad_accum=1, max_completion_length=16, eval_samples=4)

os.makedirs(OUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
# is_bf16_supported() returns True on a T4 via slow emulation, and LoRA training then crashes.
# Native bf16 needs compute capability 8.0+ (A100, L4, RTX 30xx+).
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if use_bf16 else torch.float32   # T4 has no bf16; fp32 for 0.5B fits fine
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(MODEL_ID, DEVICE, DTYPE, CFG)
# Fail fast instead of silently running for hours on CPU (e.g. Kaggle accelerator left at "None").
assert SMOKE or DEVICE == "cuda", "No GPU found: set the accelerator to a T4 GPU and restart."

## 2. Data: synthetic arithmetic with held-out OOD splits

In [ ]:
splits = rlmath.build_datasets(CFG["n_train"], CFG["n_eval"], seed=SEED)
for k, v in splits.items():
    print(f"{k:16s} n={len(v):5d}  e.g. {v[0]['expression']} = {v[0]['answer']}")
eval_splits = {k: v for k, v in splits.items() if k.startswith("test")}

## 3. Baseline: the base model before any RL

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE)

def run_eval(model, tag, out_dir=None):
    # Saves after each split and skips splits already on disk, so it resumes after a disconnect.
    path = f"{out_dir or OUT_DIR}/{tag}_eval.json"
    res = json.load(open(path)) if os.path.exists(path) else {}
    for name, rows in eval_splits.items():
        if name in res:
            print(f"[{tag}] {name}: cached greedy={res[name]['greedy_acc']:.3f}")
            continue
        t = time.time()
        res[name] = rlmath.evaluate(model, tok, rows, n_samples=CFG["eval_samples"],
                                    ks=(1, 2, 4, 8), max_new_tokens=CFG["max_completion_length"],
                                    batch_size=128)
        print(f"[{tag}] {name}: greedy={res[name]['greedy_acc']:.3f} ({time.time()-t:.0f}s)")
        rlmath.save_results(path, res)
    return res

base_res = run_eval(base, "base")
print(rlmath.summarize(base_res))

In [ ]:
# GATE: check this before training. Every response starts with "<think>" (PREFILL).
# Want: a good share of format_ok, and lenient_acc not far above greedy_acc
# (a big gap means correct math is being lost to formatting).
import collections
print(rlmath.SYSTEM_PROMPT, "\n")
for name in eval_splits:
    print(name, collections.Counter(r["style"] for r in base_res[name]["records"]))
for r in base_res["test_id"]["records"][:3]:
    print(r["expression"], "=", r["answer"], "| pred:", r["greedy_pred"]); print(r["greedy_text"][:400]); print("-"*60)
del base; torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 4. Experiments

Every experiment shares the baseline above, the same test sets and the same seed, and changes one thing:

| Experiment | Change | Hypothesis it tests |
|---|---|---|
| `default` | Run 3 config: lenient correctness + format reward, 2–3 operand training | (reference) |
| `hard_train` | Train on 3–4 operand problems | Run 3 compressed its reasoning because short training problems never rewarded long step-by-step work. If so, harder training should keep reasoning long and recover accuracy on longer expressions. **5-operand problems are the clean OOD slice** here, since 4-operand problems now appear in training. |
| `format_only` | Format reward only, no correctness reward | Control. How much of Run 3's gain comes from learning the format and becoming consistent, with no signal about whether the math is right? |
| `kl` | `beta=0.04` KL penalty toward the base model | Run 3 lost diversity (pass@8 fell). Does anchoring to the base model preserve it? |

`RUN_LIST` chooses what this session runs. Each experiment saves to its own folder and is skipped if already finished, so you can split them across sessions.

In [ ]:
EXPERIMENTS = {
    "default":     dict(),
    "hard_train":  dict(train_split="train_hard"),
    "format_only": dict(rewards=["format"]),
    "kl":          dict(beta=0.04),
    # Extra training seeds. Data and test sets stay fixed (data seed 0); only training
    # randomness changes (LoRA init, sampling during RL, batch order, eval sampling).
    "default_s1":    dict(seed=1),
    "default_s2":    dict(seed=2),
    "hard_train_s1": dict(train_split="train_hard", seed=1),
    "hard_train_s2": dict(train_split="train_hard", seed=2),
}
RUN_LIST = ["hard_train_s1", "hard_train_s2", "default_s1", "default_s2"]   # this session, in order

REWARDS = {"correct": rlmath.correctness_reward, "format": rlmath.format_reward}

In [ ]:
import gc, glob
from peft import PeftModel

def run_experiment(name):
    cfg = {**CFG, "train_split": "train", "rewards": ["correct", "format"], "seed": SEED,
           **EXPERIMENTS[name]}
    out = f"{OUT_DIR}/{name}"
    os.makedirs(out, exist_ok=True)
    json.dump({k: v for k, v in cfg.items()}, open(f"{out}/config.json", "w"), indent=1)
    if not os.path.exists(f"{out}/final_adapter"):
        train_ds = Dataset.from_list([{"prompt": rlmath.render_prompt(tok, r), "answer": r["answer"]}
                                      for r in splits[cfg["train_split"]]])
        args = GRPOConfig(
            output_dir=out,
            max_steps=cfg["max_steps"],
            learning_rate=cfg["lr"],
            lr_scheduler_type="constant_with_warmup", warmup_steps=10,
            per_device_train_batch_size=cfg["per_device_bs"],
            gradient_accumulation_steps=cfg["grad_accum"],
            num_generations=cfg["num_generations"],
            max_completion_length=cfg["max_completion_length"],
            temperature=cfg["temperature"],
            beta=cfg["beta"],
            bf16=use_bf16,
            fp16=torch.cuda.is_available() and not use_bf16,
            gradient_checkpointing=not SMOKE,
            logging_steps=1 if SMOKE else 5,
            save_steps=50, save_total_limit=None,     # keep every checkpoint: lets you evaluate step 50, 100, ... later
            report_to="none", seed=cfg["seed"], log_completions=False,
            use_cpu=DEVICE == "cpu",
        )
        peft_cfg = LoraConfig(r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"], lora_dropout=0.0,
                              target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                              "gate_proj", "up_proj", "down_proj"],
                              task_type="CAUSAL_LM")
        torch.manual_seed(cfg["seed"])
        policy = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE)
        trainer = GRPOTrainer(model=policy, reward_funcs=[REWARDS[r] for r in cfg["rewards"]],
                              args=args, train_dataset=train_ds, processing_class=tok,
                              peft_config=peft_cfg)
        trainer.train(resume_from_checkpoint=bool(glob.glob(f"{out}/checkpoint-*")))
        trainer.save_model(f"{out}/final_adapter")
        json.dump(trainer.state.log_history, open(f"{out}/train_log.json", "w"))
        model = trainer.model.merge_and_unload()
        del trainer
    else:
        print(f"[{name}] found final_adapter, skipping training")
        model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE),
                                          f"{out}/final_adapter").merge_and_unload()
    model.to(DEVICE)
    torch.manual_seed(cfg["seed"])
    res = run_eval(model, "grpo", out_dir=out)
    del model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res

SESSION_START = time.time()
results = {"base": base_res}
for name in RUN_LIST:
    print("=" * 25, name, "=" * 25)
    t = time.time()
    results[name] = run_experiment(name)
    print(f"[{name}] done in {(time.time() - t) / 60:.0f} min")

# Also pick up experiments finished in earlier sessions (same OUT_DIR).
for name in EXPERIMENTS:
    p = f"{OUT_DIR}/{name}/grpo_eval.json"
    if name not in results and os.path.exists(p):
        r = json.load(open(p))
        if all(s in r for s in eval_splits):
            results[name] = r
print("Models available for comparison:", list(results))

## 5. Results

All accuracy numbers are **lenient** (math only, independent of tags). `5-op` is accuracy on 5-operand problems in `test_ood_length`, which is out of distribution for every experiment, including `hard_train`.

In [ ]:
def five_op(res):
    v = [r["greedy_correct_lenient"] for r in res["test_ood_length"]["records"] if r["n_operands"] == 5]
    return float(np.mean(v))

KMAX = max(k for k in (1, 2, 4, 8) if k <= CFG["eval_samples"])
for name in eval_splits:
    print(f"### {name}")
    print(f"| model | greedy | pass@1 | pass@{KMAX} | format | mean chars |")
    print("|---|---|---|---|---|---|")
    for m, res in results.items():
        r = res[name]
        print(f"| {m} | {r['lenient_acc']:.3f} | {r['lenient_pass@1']:.3f} | {r[f'lenient_pass@{KMAX}']:.3f} "
              f"| {r['format_rate']:.2f} | {r['mean_chars']:.0f} |")
    print()
print("### 5-operand slice of test_ood_length (greedy, lenient)")
for m, res in results.items():
    print(f"{m:12s} {five_op(res):.3f}")

In [ ]:
# Paired tests against the base model on the same problems.
from math import comb
def mcnemar(a, b):
    n = a + b
    return min(1.0, 2 * sum(comb(n, i) for i in range(min(a, b) + 1)) / 2 ** n) if n else 1.0

for m, res in results.items():
    if m == "base":
        continue
    print(f"--- {m} vs base")
    for name in eval_splits:
        b, g = base_res[name]["records"], res[name]["records"]
        fixed  = sum(not x["greedy_correct_lenient"] and y["greedy_correct_lenient"] for x, y in zip(b, g))
        broken = sum(x["greedy_correct_lenient"] and not y["greedy_correct_lenient"] for x, y in zip(b, g))
        new    = sum(x["n_correct_lenient"] == 0 and y["n_correct_lenient"] > 0 for x, y in zip(b, g))
        lost   = sum(x["n_correct_lenient"] > 0 and y["n_correct_lenient"] == 0 for x, y in zip(b, g))
        print(f"{name:16s} greedy fixed={fixed:3d} broken={broken:3d} p={mcnemar(fixed, broken):.3g} | "
              f"coverage new={new:3d} lost={lost:3d} p={mcnemar(new, lost):.3g}")

In [ ]:
# pass@k (lenient) for every model. Curves that start high and stay flat = sharpening.
ks = [k for k in (1, 2, 4, 8) if k <= CFG["eval_samples"]]
fig, axes = plt.subplots(1, len(eval_splits), figsize=(4.6 * len(eval_splits), 3.4), sharey=True)
for ax, name in zip(np.atleast_1d(axes), eval_splits):
    for m, res in results.items():
        ax.plot(ks, [res[name][f"lenient_pass@{k}"] for k in ks], marker="o", label=m)
    ax.set_xscale("log", base=2); ax.set_xticks(ks); ax.set_xticklabels(ks)
    ax.set_title(name); ax.set_xlabel("k"); ax.grid(alpha=.3)
np.atleast_1d(axes)[0].set_ylabel("lenient pass@k"); np.atleast_1d(axes)[0].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/pass_at_k_all.png", dpi=130); plt.show()

In [ ]:
# Training curves per experiment. Watch completion length (did reasoning shrink?) and
# frac_reward_zero_std (share of groups giving no learning signal).
keys = ["reward", "completions/mean_length", "frac_reward_zero_std", "kl"]
fig, axes = plt.subplots(1, len(keys), figsize=(4.2 * len(keys), 3))
for m in results:
    p = f"{OUT_DIR}/{m}/train_log.json"
    if m == "base" or not os.path.exists(p):
        continue
    logs = json.load(open(p))
    for ax, k in zip(axes, keys):
        pts = [(l["step"], l[k]) for l in logs if k in l]
        if pts:
            ax.plot(*zip(*pts), label=m)
for ax, k in zip(axes, keys):
    ax.set_title(k); ax.set_xlabel("step"); ax.grid(alpha=.3)
axes[0].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/training_curves_all.png", dpi=130); plt.show()

## 5b. Seeds: mean ± std

Seed 0 of each config ran in earlier sessions; its numbers are entered below so every config has 3 seeds. `default` seed 0 is Run 3; `hard_train` seed 0 is from the experiments session.

In [ ]:
PRIOR = {   # seed-0 results from earlier sessions (lenient metrics)
    "default": {"test_id": dict(greedy=0.9133, p1=0.9025, p8=0.9400),
                "test_ood_length": dict(greedy=0.6300, p1=0.6058, p8=0.8033),
                "test_ood_digits": dict(greedy=0.4800, p1=0.4725, p8=0.5467),
                "five_op": 0.5897},
    "hard_train": {"test_id": dict(greedy=0.957, p1=0.928, p8=0.983),
                   "test_ood_length": dict(greedy=0.810, p1=0.772, p8=0.923),
                   "test_ood_digits": dict(greedy=0.553, p1=0.494, p8=0.707),
                   "five_op": 0.750},
}
def summary(res):
    d = {s: dict(greedy=res[s]["lenient_acc"], p1=res[s]["lenient_pass@1"],
                 p8=res[s][f"lenient_pass@{KMAX}"]) for s in eval_splits}
    d["five_op"] = five_op(res)
    return d

groups = {}
for cfg_name in PRIOR:
    runs = [PRIOR[cfg_name]] + [summary(results[f"{cfg_name}_s{i}"]) for i in (1, 2)
                                if f"{cfg_name}_s{i}" in results]
    groups[cfg_name] = runs
base_sum = summary(base_res)

for split in eval_splits:
    print(f"### {split}  (n seeds in brackets)")
    print("| model | greedy | pass@1 | pass@8 |")
    print("|---|---|---|---|")
    b = base_sum[split]
    print(f"| base | {b['greedy']:.3f} | {b['p1']:.3f} | {b['p8']:.3f} |")
    for g, runs in groups.items():
        cells = []
        for m in ("greedy", "p1", "p8"):
            v = np.array([r[split][m] for r in runs])
            cells.append(f"{v.mean():.3f} ± {v.std(ddof=1) if len(v) > 1 else 0:.3f}")
        print(f"| {g} [{len(runs)}] | " + " | ".join(cells) + " |")
    print()
print("### 5-operand slice (greedy)")
print(f"base        {base_sum['five_op']:.3f}")
for g, runs in groups.items():
    v = np.array([r["five_op"] for r in runs])
    print(f"{g:11s} {v.mean():.3f} ± {v.std(ddof=1) if len(v) > 1 else 0:.3f}  (seeds: {np.round(v, 3).tolist()})")

## 5c. pass@64: is "newly solvable" real, or just more tries?

pass@8 can't tell "RL taught it" from "the base model would get there with more tries". This compares the base model and `hard_train_s1` with 64 samples per problem on 100 problems from each OOD split. Skipped automatically if the session has run long, so the seed results above are never put at risk.

In [ ]:
PASS64_SPLITS = ["test_ood_length", "test_ood_digits"]
PASS64_N_PROBLEMS = 100 if not SMOKE else 4
PASS64_SAMPLES = 64 if not SMOKE else 4
ks64 = [k for k in (1, 4, 16, 64) if k <= PASS64_SAMPLES]
elapsed_h = (time.time() - SESSION_START) / 3600

def pass64_eval(model, tag):
    path = f"{OUT_DIR}/pass64_{tag}.json"
    res = json.load(open(path)) if os.path.exists(path) else {}
    for sp in PASS64_SPLITS:
        if sp in res:
            continue
        torch.manual_seed(SEED)
        r = rlmath.evaluate(model, tok, eval_splits[sp][:PASS64_N_PROBLEMS], n_samples=PASS64_SAMPLES,
                            ks=tuple(ks64), max_new_tokens=CFG["max_completion_length"],
                            batch_size=128, greedy=False)
        res[sp] = {k: v for k, v in r.items() if k.startswith("lenient_pass@")}
        res[sp]["n_correct_lenient"] = [x["n_correct_lenient"] for x in r["records"]]
        rlmath.save_results(path, res)
        print(f"[pass64 {tag}] {sp}: " + ", ".join(f"{k}={res[sp][f'lenient_pass@{k}']:.3f}" for k in ks64))
    return res

if elapsed_h > 8 and not SMOKE:
    print(f"Skipping pass@64: session already at {elapsed_h:.1f} h")
elif not os.path.exists(f"{OUT_DIR}/hard_train_s1/final_adapter"):
    print("Skipping pass@64: hard_train_s1 adapter not found in this session")
else:
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE)
    p64_base = pass64_eval(m, "base"); del m
    m = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE),
                                  f"{OUT_DIR}/hard_train_s1/final_adapter").merge_and_unload().to(DEVICE)
    p64_hard = pass64_eval(m, "hard_train_s1"); del m
    gc.collect()
    for sp in PASS64_SPLITS:
        nb = np.array(p64_base[sp]["n_correct_lenient"]); nh = np.array(p64_hard[sp]["n_correct_lenient"])
        print(f"{sp}: solved at least once in {PASS64_SAMPLES}: base={np.mean(nb > 0):.2f} hard={np.mean(nh > 0):.2f} | "
              f"hard-only={int(((nh > 0) & (nb == 0)).sum())} base-only={int(((nb > 0) & (nh == 0)).sum())}")

## 6. How to read the results

- **`hard_train`**: if mean chars on `test_ood_length` stays near the base model's (~340) and the 5-operand accuracy holds or improves, the reasoning-compression explanation for Run 3 is supported. If reasoning still shrinks, brevity comes from GRPO itself, not the training distribution.
- **`format_only`**: if pass@1 rises almost as much as in `default`, most of RL's apparent gain is format and consistency, not math. If it stays near the base model, the correctness signal is doing real work.
- **`kl`**: compare pass@8 with `default`. If it stays near the base model while pass@1 still rises, the KL penalty preserved diversity.
- Single seed per experiment: treat differences under ~3 points as noise unless the paired test says otherwise.

Each `checkpoint-50`, `-100`, ... folder is kept, so you can later evaluate how these effects develop over training.